# Event-SAE on Google Colab Pro (L4 GPU)

Reproduce the paper result: OpenVLA + LIBERO-Goal, baseline **68.0%** -> zero out top 5 event-aligned features -> **48.8%** (random features: 68.7%).

Paper: Event-Grounded Sparse Autoencoders for Vision-Language-Action Policies (NeurIPS 2026). Code: https://github.com/xc-j/Event-SAE

## BEFORE YOU RUN ANYTHING — pick the L4 GPU
1. Top menu: **Runtime → Change runtime type**
2. Hardware accelerator: **GPU**
3. GPU class: **L4** (22.5 GB)
4. Save, then run the cells top to bottom.

Why L4: the 7B OpenVLA model needs about 14 GB in bf16. A T4 (16 GB) is too tight; the L4's 22.5 GB fits it comfortably, so there is NO 8-bit fallback in this notebook.

Colab Pro keeps the runtime running in the background even if you close the tab or your internet drops, so a long cell keeps working. An L4 uses about 1.7 compute units an hour (measured by users, not published by Google), so your 100 monthly units buy roughly 58 hours of L4 — check the rate on Colab's resource monitor.

Run the cells top to bottom. Do NOT skip the patch cells (Step 3b) or the render test — they are the fixes learned from the Kaggle run.

## Step 0 — GPU check

Check the GPU is an L4 with about 22.5 GB. If this cell says T4 or anything else, stop and change the runtime type first (Runtime → Change runtime type → GPU class L4).

In [1]:
!nvidia-smi
import sys, torch
print("Python:", sys.version)
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    free = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(f"GPU {i}: {p.name} | total: {p.total_memory/1024**3:.1f} GiB | free: {free:.1f} GiB")
    if "L4" in p.name:
        print("  -> L4 confirmed. Good.")
    else:
        print("  -> WARNING: this is not an L4. Stop and change the runtime type to L4 first.")

Mon Nov 23 16:30:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Clone the code

In [2]:
import os, subprocess
REPO = "/content/Event-SAE"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/xc-j/Event-SAE.git", REPO], check=True)
else:
    print("Event-SAE already cloned")
LIB = REPO + "/external/LIBERO"
if not os.path.isdir(LIB):
    subprocess.run(["git", "clone", "https://github.com/Lifelong-Robot-Learning/LIBERO.git", LIB], check=True)
    subprocess.run(["git", "-C", LIB, "checkout", "f78abd6"], check=True)
else:
    print("LIBERO already cloned")
# Files the README asks us to create (LIBERO expects these __init__.py files)
open(LIB + "/libero/__init__.py", "a").close()
open(LIB + "/libero/lifelong/models/modules/__init__.py", "a").close()
print("Done. Repo contents:")
print(os.listdir(REPO))

Done. Repo contents:
['.git', 'external', 'event_sae', 'third_party', 'LICENSE', 'pyproject.toml', 'scripts', 'docs', 'README.md', 'configs', '.gitignore']


## Step 2 — Install (Colab-adapted)

Colab's image is Python 3.12 + a recent torch (2.6 or newer) + CUDA 12.8. The repo pins torch==2.2.0, numpy==1.26.4 and tensorflow==2.16.1 — those have no wheels for this Python, so they CANNOT be installed here. We keep Colab's torch and numpy and install the rest.

We also skip flash-attn: the README's wheel is built for torch 2.2 + CUDA 12.2 + Python 3.10 and cannot install here. Every rollout command below uses `--attention sdpa` instead.

The installs are SPLIT into separate cells on purpose (learned on Kaggle — one big line fails as a whole when a single package fails):
- `robosuite` installs with `--no-build-isolation` — its dependency `egl_probe` cannot build its wheel here, and without this flag pip aborts the whole install.
- `robomimic` installs ALONE with `--no-deps` — it pulls `robosuite`, which pulls `egl_probe`, so it must never sit in a line with other packages.
- `mujoco` is pinned to 3.5.0 AFTER robosuite, because robosuite pulls the newest mujoco.
- `egl_probe` is skipped entirely — nothing in the run needs it.

In [3]:
%cd /content/Event-SAE
# Install the two packages themselves without dependencies (their requirements
# files would downgrade torch — LIBERO's own README pins torch 1.11).
!pip install -q --no-deps -e external/LIBERO
!pip install -q --no-deps -e .

/content/Event-SAE
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for libero (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for event-sae (pyproject.toml) ... done


In [4]:
# The repo's other dependencies (robosuite, mujoco and robomimic are NOT in this
# line — they each get their own cell below, for the reasons in the Step 2 notes).
!pip install -q transformers==4.48.1 tokenizers==0.21.2 timm==0.9.16 accelerate==1.12.0 einops==0.8.1 sentencepiece tqdm pyyaml scipy==1.15.3 scikit-learn==1.7.2 bddl==1.0.1 gym==0.25.2 numba==0.64.0 easydict cloudpickle future matplotlib opencv-python==4.11.0.86 "imageio[ffmpeg]" pillow ninja==1.13.0 "cmake<4" "setuptools<82"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
# robosuite alone, with --no-build-isolation: its dependency egl_probe cannot
# build a wheel in this image, and a normal install aborts on it.
!pip install -q "robosuite==1.4.0" --no-build-isolation

In [6]:
# robosuite pulled the newest mujoco — pin it back to the version this repo needs.
!pip install -q "mujoco==3.5.0"

In [7]:
# robomimic ALONE with --no-deps. Never put it in a line with other packages:
# it pulls robosuite, robosuite pulls egl_probe, and the whole line aborts.
!pip install -q "robomimic==0.3.0" --no-deps

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
# TensorFlow is only used for image crop/resize. The pinned 2.16.1 has no wheel
# for this Python, so install the current version instead.
!pip install -q tensorflow
import importlib
for pkg in ["event_sae", "transformers", "robosuite", "mujoco", "tensorflow", "bddl", "gym", "robomimic"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg, "OK", getattr(m, "__version__", ""))
    except Exception as e:
        print(pkg, "FAILED:", type(e).__name__, e)
# If a package above says FAILED, run its install cell again alone and paste the
# exact error. If a package you just installed still cannot be found, use
# Runtime -> Restart runtime and run the cells from the top again.

event_sae OK 


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.13/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


transformers OK 4.48.1
robosuite OK 1.4.0
mujoco OK 3.5.0
tensorflow OK 2.21.0
bddl OK 
gym OK 0.25.2
robomimic OK 0.3.0


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Step 3 — Download the pretrained SAE checkpoint + the OpenVLA policy

The SAE checkpoint is small. The OpenVLA policy is ~14 GB — this cell downloads it now so a network error shows up here, not in the middle of a run. Both go to the notebook's cache.

In [9]:
import os
from huggingface_hub import snapshot_download
SAE_DIR = snapshot_download("mr-cabbage/event-sae-libero",
    allow_patterns=["openvla/libero_goal/ae.pt", "openvla/libero_goal/config.json"])
SAE_CKPT = os.path.join(SAE_DIR, "openvla/libero_goal/ae.pt")
print("SAE checkpoint:", SAE_CKPT)
print("ae.pt exists:", os.path.isfile(SAE_CKPT))
# OpenVLA policy — checkpoint + revision are from configs/models/openvla_libero_goal.yaml
MODEL_DIR = snapshot_download("openvla/openvla-7b-finetuned-libero-goal",
    revision="fa5ae1e7509348889295bba8e08621d8b55e9baf")
print("OpenVLA model dir:", MODEL_DIR)
!df -h /content

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

openvla/libero_goal/ae.pt:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


SAE checkpoint: /root/.cache/huggingface/hub/models--mr-cabbage--event-sae-libero/snapshots/d3cc3ad4884b4b84a5c1fb7b37bdbb2f64760e14/openvla/libero_goal/ae.pt
ae.pt exists: True


... (14 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

OpenVLA model dir: /root/.cache/huggingface/hub/models--openvla--openvla-7b-finetuned-libero-goal/snapshots/fa5ae1e7509348889295bba8e08621d8b55e9baf
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   18G   96G  16% /


## Step 3b — Patches (fixes learned on Kaggle, now the default)

Two one-line patches, both already proven on the Kaggle run:
1. **LIBERO `torch.load` patch** — PyTorch 2.6+ refuses to load LIBERO's initial-states file (`weights_only` now defaults to True). The file ships with LIBERO's own official repo, so loading it with `weights_only=False` is safe.
2. **Rendering patch** — the code sets `MUJOCO_GL` with `setdefault`, which does NOTHING if the container already exports it (Kaggle exported `egl`, so the first patch never took effect). We force a direct assignment to `osmesa` (software rendering) and set `PYOPENGL_PLATFORM=osmesa` too — MuJoCo refuses osmesa unless both are set.

In [10]:
%cd /content/Event-SAE
# Patch LIBERO: torch.load needs weights_only=False for its own init-states file.
!sed -i '164s/torch.load(init_states_path)/torch.load(init_states_path, weights_only=False)/' external/LIBERO/libero/libero/benchmark/__init__.py
!grep -n "weights_only" external/LIBERO/libero/libero/benchmark/__init__.py
# You must see line 164 with weights_only=False. If grep prints nothing, STOP and report it.

/content/Event-SAE
164:        init_states = torch.load(init_states_path, weights_only=False)


In [11]:
%cd /content/Event-SAE
# Force OSMesa rendering in the rollout code (direct assignment, NOT setdefault).
!sed -i '30s/os.environ.setdefault("MUJOCO_GL", "egl")/os.environ["MUJOCO_GL"] = "osmesa"/' event_sae/rollout.py
!sed -i '30a os.environ["PYOPENGL_PLATFORM"] = "osmesa"' event_sae/rollout.py
!grep -n -E "MUJOCO_GL|PYOPENGL_PLATFORM" event_sae/rollout.py
# You must see line 30 as os.environ["MUJOCO_GL"] = "osmesa" and line 31 as
# os.environ["PYOPENGL_PLATFORM"] = "osmesa". If not, STOP and report the grep output.
from pathlib import Path
src = Path("event_sae/rollout.py").read_text()
print("rollout.py MUJOCO_GL patch:", "OK" if 'os.environ["MUJOCO_GL"] = "osmesa"' in src else "CHECK FAILED — report the grep output above")
print("rollout.py PYOPENGL patch:", "OK" if 'os.environ["PYOPENGL_PLATFORM"] = "osmesa"' in src else "CHECK FAILED — report the grep output above")
# Set the same two variables in this notebook's environment so they reach every
# subprocess the rollout starts.
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"], "| PYOPENGL_PLATFORM =", os.environ["PYOPENGL_PLATFORM"])

/content/Event-SAE
30:    os.environ["MUJOCO_GL"] = "osmesa"
31:os.environ["PYOPENGL_PLATFORM"] = "osmesa"
rollout.py MUJOCO_GL patch: OK
rollout.py PYOPENGL patch: OK
MUJOCO_GL = osmesa | PYOPENGL_PLATFORM = osmesa


## Render test — run this BEFORE Step 4

MuJoCo rendering itself must work in this container. This cell renders one tiny 64x64 frame in a fresh process with both variables set to osmesa. It must print `RENDER OK (64, 64, 3)`. If it fails, STOP and report the exact error — do not run Step 4.

In [12]:
%%writefile /tmp/render_test.py
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import mujoco
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom type="sphere" size="0.1"/></worldbody></mujoco>')
d = mujoco.MjData(m)
r = mujoco.Renderer(m, height=64, width=64)
r.update_scene(d)
pix = r.render()
print("RENDER OK", pix.shape)

Writing /tmp/render_test.py


In [13]:
!MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa python /tmp/render_test.py

Traceback (most recent call last):
  File "/tmp/render_test.py", line 4, in <module>
    import mujoco
  File "/usr/local/lib/python3.13/dist-packages/mujoco/__init__.py", line 62, in <module>
    from mujoco.gl_context import *
  File "/usr/local/lib/python3.13/dist-packages/mujoco/gl_context.py", line 38, in <module>
    from mujoco.osmesa import GLContext as _GLContext
  File "/usr/local/lib/python3.13/dist-packages/mujoco/osmesa/__init__.py", line 31, in <module>
    from OpenGL import GL
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/GL/__init__.py", line 4, in <module>
    from OpenGL.GL.VERSION.GL_1_1 import *
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/GL/VERSION/GL_1_1.py", line 14, in <module>
    from OpenGL.raw.GL.VERSION.GL_1_1 import *
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/raw/GL/VERSION/GL_1_1.py", line 7, in <module>
    from OpenGL.raw.GL import _errors
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/raw/GL/_errors.py", line 4

## Step 4 — Collect 50 LIBERO-Goal episodes (this is the long cell)

Runs the OpenVLA policy in the LIBERO simulator and saves layer-31 activations. Each episode reloads the 7B model in its own process. On the L4 this is roughly 2-3x faster than the T4 estimate — expect a few hours for all 50 episodes. Colab Pro keeps the runtime running in the background even if you close the tab or your internet drops.

The cell deletes `outputs/rollout` first: the script refuses to reuse an output folder, and a failed run leaves an empty one behind. If the 12-hour session ends first, the finished episode folders stay — the Step 10 cell can still use them.

In [ ]:
%cd /content/Event-SAE
!rm -rf outputs/rollout
!python scripts/rollout_native.py --backend openvla \
  --model-config configs/models/openvla_libero_goal.yaml \
  --collect-config configs/collection/openvla.yaml \
  --task-ids 0-9 --initial-states 0-4 --output outputs/rollout --attention sdpa

/content/Event-SAE
configuration_prismatic.py: 5.87kB [00:00, 24.5MB/s]
modeling_prismatic.py: 26.1kB [00:00, 70.8MB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.48.1` and `tokenizers==0.21.2`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.
Loading checkpoint shards: 100% 4/4 [00:03<00:00,  1.27it/s]
processing_prismatic.py: 12.7kB [00:00, 9.35MB/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved wit

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Step 5 — Extract top-k features with the pretrained SAE (NO training)

We skip SAE training and use the paper's checkpoint downloaded in Step 3.

In [ ]:
%cd /content/Event-SAE
import os
from huggingface_hub import snapshot_download
SAE_DIR = snapshot_download("mr-cabbage/event-sae-libero",
    allow_patterns=["openvla/libero_goal/ae.pt", "openvla/libero_goal/config.json"])
SAE_CKPT = os.path.join(SAE_DIR, "openvla/libero_goal/ae.pt")
print("SAE checkpoint:", SAE_CKPT)
!python scripts/extract_topk.py \
  --dense-dir outputs/rollout/sae_activations/resid_post \
  --sae-checkpoint {SAE_CKPT} \
  --layer-idx 31 --topk 64 --output-dir outputs/topk

/content/Event-SAE


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

SAE checkpoint: /root/.cache/huggingface/hub/models--mr-cabbage--event-sae-libero/snapshots/d3cc3ad4884b4b84a5c1fb7b37bdbb2f64760e14/openvla/libero_goal/ae.pt
Encoded layer_31_shard_000000_episode_00000.pt → shard_000000.pt  rows=32490
[W1123 11:56:58.080062584 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 6574571520 bytes (free: 1698693120, total: 23659151360).
Encoded layer_31_shard_000000_episode_00001.pt → shard_000001.pt  rows=50154
[W1123 11:57:00.263716577 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 4894752768 bytes (free: 448790528, total: 23659151360).
[W1123 11:57:01.113461017 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 4894752768 bytes (free: 4708106240, total: 23659151360).
Traceback (most recent call last):
  File "/content/Event-SAE/scripts/extract_topk.py", line 249, in <module>
    main()
    ~~~~^^
 

## Step 6 — Keyframes and events

In [ ]:
%cd /content/Event-SAE
!python scripts/extract_keyframes.py \
  --trajectory-records-path outputs/rollout/trajectory_records.jsonl \
  --output-dir outputs/keyframes
!python scripts/extract_keyframe_media.py \
  --waypoint-summary-path outputs/keyframes/waypoint_summary.json \
  --output-dir outputs/events

/content/Event-SAE
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.13/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Traceback (most recent call last):
  File "/content/Event-SAE/scripts/extract_keyframes.py", line 14, in <module>
    from event_sae.keyframes import (
    ...<5 lines>...
    )
  File "/content/Event-SAE/event_sae/keyframes/__init__.py", line 1, in <module>
    from event_sae.keyframes.extract import (
    ...<6 lines>...
    )
  File "/content/Event-SAE/event_sae/keyframes/extract.py", line 19, in <module>
    from waypoint_extraction import dp_waypoint_selection
  File "/content/Event-SAE/third_party/awe/waypoint_extraction/__init__.py", line 1, in <module>
    from .extract_waypoints import dp_waypoint_selection
  File "/content/Event-SAE/third_party/awe/waypoint_extraction/extract_waypoin

In [ ]:
%cd /content/Event-SAE
# SigLIP downloads on first use — Internet must stay ON.
!python scripts/build_event_features.py --samples-path outputs/events/samples.jsonl
!python scripts/cluster_events.py --event-features-path outputs/events/event_features.jsonl
# NOTE: annotate_clusters.py is skipped — it needs a Gemini API key and is optional.

/content/Event-SAE
I0000 00:00:1795425743.046164    2667 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795425745.649337    2667 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795425745.649919    2667 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Traceback (most recent call last):
  File "/content/Event-SAE/scripts/build_event_features.py", line 65, in <module>
    main()
    ~~~~^^
  File "/content/Event-SAE/scripts/build_event_features.py", line 54, in main
    build_event_features(
    ~~~~~~~~~~~~~~~~~~~~^
        samples_path=sampl

## Step 7 — Score features against events and rank them

In [ ]:
%cd /content/Event-SAE
!python scripts/score_cluster_features.py \
  --topk-run-dir outputs/topk \
  --event-features-path outputs/events/event_features.jsonl \
  --cluster-assignments-path outputs/events/clusters/cluster_assignments.jsonl \
  --clusters-path outputs/events/clusters/clusters.jsonl \
  --prompt-records-path outputs/rollout/prompt_records.jsonl \
  --output-path outputs/scores.pt
!python scripts/build_feature_rankings.py \
  --scores-pt outputs/scores.pt \
  --topk-run-dir outputs/topk \
  --top-k 5 --seed 0 --output-dir outputs/rankings
import json
print("Top event-aligned candidates (rank 1 = the feature we will zero out):")
with open("outputs/rankings/candidates.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if row["ranking"] == "event_aligned" and row["rank"] <= 5:
            print("  rank", row["rank"], "feature_id =", row["feature_id"])

/content/Event-SAE
Traceback (most recent call last):
  File "/content/Event-SAE/scripts/score_cluster_features.py", line 68, in <module>
    main()
    ~~~~^^
  File "/content/Event-SAE/scripts/score_cluster_features.py", line 52, in main
    summary = score_cluster_features(
        topk_run_dir=Path(args.topk_run_dir),
    ...<8 lines>...
        prompt_records_path=Path(args.prompt_records_path) if args.prompt_records_path else None,
    )
  File "/content/Event-SAE/event_sae/scoring/score_matrix.py", line 498, in score_cluster_features
    manifest = _load_manifest(topk_run_dir)
  File "/content/Event-SAE/event_sae/scoring/score_matrix.py", line 293, in _load_manifest
    raise FileNotFoundError(f"No token_topk_sparse_v1 manifest under {topk_run_dir}")
FileNotFoundError: No token_topk_sparse_v1 manifest under /content/Event-SAE/outputs/topk
Traceback (most recent call last):
  File "/content/Event-SAE/scripts/build_feature_rankings.py", line 153, in <module>
    main()
    ~~~~^^


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/rankings/candidates.jsonl'

## Step 8 — Write the three steering configs

Baseline = no intervention. Event = zero out the rank-1 event-aligned feature. Random = zero out a random alive feature (the control).

In [ ]:
%cd /content/Event-SAE
import os, yaml
from huggingface_hub import snapshot_download
SAE_DIR = snapshot_download("mr-cabbage/event-sae-libero",
    allow_patterns=["openvla/libero_goal/ae.pt", "openvla/libero_goal/config.json"])
SAE_CKPT = os.path.join(SAE_DIR, "openvla/libero_goal/ae.pt")
base = {"sae_checkpoint": SAE_CKPT, "layer_idx": 31, "feature_alpha": 0.0, "hook_start_step": 0}
with open("configs/steering/colab_native.yaml", "w") as f:
    yaml.safe_dump({"mode": "native"}, f)
with open("configs/steering/colab_event.yaml", "w") as f:
    yaml.safe_dump({**base, "mode": "feature",
                    "candidates": "outputs/rankings/candidates.jsonl",
                    "ranking": "event_aligned", "rank": 1}, f)
with open("configs/steering/colab_random.yaml", "w") as f:
    yaml.safe_dump({**base, "mode": "feature",
                    "candidates": "outputs/rankings/candidates.jsonl",
                    "ranking": "random_alive", "rank": 1}, f)
print("Wrote colab_native.yaml, colab_event.yaml, colab_random.yaml")
print("SAE checkpoint used:", SAE_CKPT)

## Step 9 — Baseline, event-aligned and random batches (20 episodes each)

20 episodes = tasks 0-9 x initial states 0-1. This is a direction check, not the exact paper number (the paper used 100 episodes). Each batch can take hours — run them in separate sessions if needed. Use a FRESH output folder name if you re-run a batch (the scripts refuse to reuse one).

In [ ]:
%cd /content/Event-SAE
!python scripts/rollout_steered.py --backend openvla \
  --model-config configs/models/openvla_libero_goal.yaml \
  --steer-config configs/steering/colab_native.yaml \
  --task-ids 0-9 --initial-states 0-1 --seed 0 \
  --output outputs/baseline-20 --attention sdpa

In [ ]:
%cd /content/Event-SAE
!python scripts/rollout_steered.py --backend openvla \
  --model-config configs/models/openvla_libero_goal.yaml \
  --steer-config configs/steering/colab_event.yaml \
  --task-ids 0-9 --initial-states 0-1 --seed 0 \
  --output outputs/steered-event-20 --attention sdpa

In [ ]:
%cd /content/Event-SAE
!python scripts/rollout_steered.py --backend openvla \
  --model-config configs/models/openvla_libero_goal.yaml \
  --steer-config configs/steering/colab_random.yaml \
  --task-ids 0-9 --initial-states 0-1 --seed 0 \
  --output outputs/steered-random-20 --attention sdpa

## Step 10 — Results

In [ ]:
%cd /content/Event-SAE
import json
from pathlib import Path
def read_summary(outdir):
    s = Path(outdir) / "summary.json"
    if s.is_file():
        d = json.loads(s.read_text())
        return d["num_episodes"], d["success_rate"]
    # Fallback: aggregate per-episode result.json files (run was cut short)
    eps = sorted(Path(outdir).glob("episodes/*/result.json"))
    if not eps:
        return None, None
    succ = sum(json.loads(p.read_text())["success"] for p in eps)
    return len(eps), succ / len(eps)
print("TARGET (paper Table 3, OpenVLA, LIBERO-Goal, 100 episodes):")
print("  baseline 68.0%  |  event-aligned zero-out 48.8%  |  random zero-out 68.7%")
print()
for name, outdir in [("Baseline (native)", "outputs/baseline-20"),
                     ("Event-aligned zero-out", "outputs/steered-event-20"),
                     ("Random zero-out (control)", "outputs/steered-random-20")]:
    n, rate = read_summary(outdir)
    if n is None:
        print(name + ": NO RESULTS YET - run that batch first")
    else:
        print(f"{name}: {rate:.1%}  ({n} episodes)")
print()
print("20 episodes is a direction check. If event-aligned is clearly below")
print("baseline and random is close to baseline, the reproduction worked.")

## Optional — full 100-episode runs (the paper's numbers)

Same three commands with `--task-ids 0-9 --initial-states 0-9` and fresh output folders (`outputs/baseline-100`, `outputs/steered-event-100`, `outputs/steered-random-100`). Each 100-episode batch is several GPU-hours. On the L4, all three together are roughly 8–15 GPU-hours — about 14–26 compute units at ~1.7 units an hour, well inside one month's 100 units — but they will NOT fit in one 12-hour Pro session. Split them across sessions, then re-run the Step 10 cell with the `-100` folder names.

## Download your results

Colab's files disappear when the runtime is recycled, so zip everything before you close the notebook. The zip appears in the file browser on the left — right-click it and Download.

In [ ]:
%cd /content/Event-SAE
!zip -r event-sae-results.zip outputs/
!ls -lh event-sae-results.zip
print("Download event-sae-results.zip from the file browser on the left.")

In [14]:
!sed -n '25,40p' /content/Event-SAE/event_sae/rollout.py
!cd /content/Event-SAE && git status --short

    if not (package / "bddl_files").is_dir():
        raise ValueError(f"Not a LIBERO checkout: {root}")
    config_dir = output / "libero_config"
    config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["LIBERO_CONFIG_PATH"] = str(config_dir.resolve())
    os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
    paths = {
        "benchmark_root": str(package),
        "bddl_files": str(package / "bddl_files"),
        "init_states": str(package / "init_files"),
        "datasets": str(root.resolve() / "libero" / "datasets"),
        "assets": str(package / "assets"),
    }
    (config_dir / "config.yaml").write_text(yaml.safe_dump(paths))

 M event_sae/rollout.py


In [15]:
!cd /content/Event-SAE && git checkout -- event_sae/rollout.py
import ast
ast.parse(open('/content/Event-SAE/event_sae/rollout.py').read())
print('rollout.py OK')

rollout.py OK


In [16]:
!sed -i '30s/os.environ.setdefault("MUJOCO_GL", "egl")/os.environ["MUJOCO_GL"] = "egl"/' /content/Event-SAE/event_sae/rollout.py
!grep -n "MUJOCO_GL" /content/Event-SAE/event_sae/rollout.py

30:    os.environ["MUJOCO_GL"] = "egl"


In [17]:
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"; print(os.environ["MUJOCO_GL"], os.environ["PYOPENGL_PLATFORM"])

egl egl


In [18]:
%%writefile /tmp/render_test_egl.py
import os
os.environ["MUJOCO_GL"]="egl"
os.environ["PYOPENGL_PLATFORM"]="egl"
import mujoco
m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom type="sphere" size="0.1"/></worldbody></mujoco>')
d=mujoco.MjData(m)
r=mujoco.Renderer(m,height=64,width=64)
r.update_scene(d)
pix=r.render()
print("RENDER OK",pix.shape)

Writing /tmp/render_test_egl.py


In [19]:
!MUJOCO_GL=egl PYOPENGL_PLATFORM=egl python /tmp/render_test_egl.py

RENDER OK (64, 64, 3)
Exception ignored in: <function Renderer.__del__ at 0x7f520cc379c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/mujoco/renderer.py", line 340, in __del__
  File "/usr/local/lib/python3.13/dist-packages/mujoco/renderer.py", line 326, in close
  File "/usr/local/lib/python3.13/dist-packages/mujoco/egl/__init__.py", line 133, in free
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/platform/baseplatform.py", line 487, in __call__
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/error.py", line 230, in glCheckError
OpenGL.raw.EGL._errors.EGLError: <exception str() failed>
Exception ignored in: <function GLContext.__del__ at 0x7f5221caf6a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/mujoco/egl/__init__.py", line 138, in __del__
  File "/usr/local/lib/python3.13/dist-packages/mujoco/egl/__init__.py", line 133, in free
  File "/usr/local/lib/python3.13/dist-packages/OpenGL/error.py

In [ ]:
!rm -rf /content/Event-SAE/outputs/rollout

In [ ]:
!nvidia-smi

Mon Nov 23 11:52:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   56C    P0             28W /   72W |     194MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ps aux | grep -E "rollout|openvla" | grep -v grep

In [ ]:
!nvidia-smi

Mon Nov 23 11:53:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   55C    P0             28W /   72W |     194MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!rm -rf /content/Event-SAE/outputs/topk

In [ ]:
!cp /content/Event-SAE/scripts/extract_topk.py /content/Event-SAE/scripts/extract_topk_chunked.py

In [ ]:
path='/content/Event-SAE/scripts/extract_topk_chunked.py'; src=open(path).read(); old='        with torch.no_grad():\n            encoded = sae.encode(dense[keep].to(device))\n            values, indices = torch.topk(encoded, k=args.topk, dim=-1)\n            values = values.float().cpu()\n            indices = indices.to(torch.int32).cpu()\n'; new='        with torch.no_grad():\n            kept = dense[keep]\n            _vals, _idxs = [], []\n            for _i in range(0, kept.shape[0], 8192):\n                _enc = sae.encode(kept[_i:_i + 8192].to(device))\n                _v, _ix = torch.topk(_enc, k=args.topk, dim=-1)\n                _vals.append(_v.float().cpu())\n                _idxs.append(_ix.to(torch.int32).cpu())\n                del _enc, _v, _ix\n                torch.cuda.empty_cache()\n            values = torch.cat(_vals, dim=0)\n            indices = torch.cat(_idxs, dim=0)\n'; open(path,'w').write(src.replace(old,new)) if old in src else print('OLD BLOCK NOT FOUND'); print('PATCHED') if old in src else None

PATCHED


In [ ]:
!python -c "import ast; ast.parse(open('/content/Event-SAE/scripts/extract_topk_chunked.py').read()); print('chunked OK')"

chunked OK


In [ ]:
!rm -rf /content/Event-SAE/outputs/topk

In [ ]:
%cd /content/Event-SAE
import os
from huggingface_hub import snapshot_download
SAE_DIR=snapshot_download("mr-cabbage/event-sae-libero", allow_patterns=["openvla/libero_goal/ae.pt","openvla/libero_goal/config.json"])
SAE_CKPT=os.path.join(SAE_DIR,"openvla/libero_goal/ae.pt")
print("SAE checkpoint:",SAE_CKPT)
!python scripts/extract_topk_chunked.py --dense-dir outputs/rollout/sae_activations/resid_post --sae-checkpoint {SAE_CKPT} --layer-idx 31 --topk 64 --output-dir outputs/topk

/content/Event-SAE


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

SAE checkpoint: /root/.cache/huggingface/hub/models--mr-cabbage--event-sae-libero/snapshots/d3cc3ad4884b4b84a5c1fb7b37bdbb2f64760e14/openvla/libero_goal/ae.pt
Encoded layer_31_shard_000000_episode_00000.pt → shard_000000.pt  rows=32490
Encoded layer_31_shard_000000_episode_00001.pt → shard_000001.pt  rows=50154
Encoded layer_31_shard_000000_episode_00002.pt → shard_000002.pt  rows=37335
Encoded layer_31_shard_000000_episode_00003.pt → shard_000003.pt  rows=50154
Encoded layer_31_shard_000000_episode_00004.pt → shard_000004.pt  rows=50154
Encoded layer_31_shard_000000_episode_00005.pt → shard_000005.pt  rows=22800
Encoded layer_31_shard_000000_episode_00006.pt → shard_000006.pt  rows=22515
Encoded layer_31_shard_000000_episode_00007.pt → shard_000007.pt  rows=22800
Encoded layer_31_shard_000000_episode_00008.pt → shard_000008.pt  rows=25080
Encoded layer_31_shard_000000_episode_00009.pt → shard_000009.pt  rows=23940
Encoded layer_31_shard_000000_episode_00010.pt → shard_000010.pt  rows=